<a href="https://colab.research.google.com/github/Bilra-Hkr/BigData26_B_2411533007_DerielChaerahman/blob/main/Praktikum2/BD_B_P02_Latihan_2411533007_DerielChaerahman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Install library faker dan import library

In [1]:
pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 29.2 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from faker import Faker
import random

# 2. Membuat Dataset Sintetis

In [3]:
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional
    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


Note : Output deterministik (515 baris), karena menggunakan SEED=42.

# 3. Deteksi Missing Value (Anomali data)

In [4]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64


# 4. Penanganan Duplicate Data

In [5]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")
print("Jumlah baris setelah penanganan missing value:", len(df))

Jumlah baris setelah penanganan missing value: 495


In [6]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


# 5. Koreksi tipe data dan  Standardisasi Format

In [7]:
# a. Standarisasi teks kategorikal (category, payment_method, shipping_city) :
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

In [8]:
#  b. Koreksi tipe data pada kolom price (dari teks bercampur simbol, menjadi numerik):
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)


In [9]:
#  c. Standardisasi format tanggal ke YYYY-MM-DD:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

In [10]:
# d. Finalisasi tipe data
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

# 6.  Ekspor Dataset Bersih

In [11]:
# Pengecekan ulang
print(df.isnull().sum())

transaction_id        0
customer_name         0
product_name          0
category              0
price                 0
quantity              0
payment_method        0
transaction_date      0
shipping_city         0
rating              118
dtype: int64


In [12]:
# Simpan CSV
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


# Latihan

In [13]:
# Menambahkan kolom untuk memeriksa apakah ada harga tidak valid
df["is_valid_price"] = df["price"] > 0

print(f"Distribusi is_valid_price = {df['is_valid_price'].value_counts().to_dict()}")

Distribusi is_valid_price = {True: 490}


In [14]:
df.head()

,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating,is_valid_price
0,TRX00490,"Baktiadi Napitupulu, S.H.",Aspernatur Basic,Rumah Tangga,120000.0,2,Transfer Bank,2026-09-19,Bau-Bau,1.0,True
1,TRX00430,Faizah Kusumo,Voluptates Max,Olahraga,250000.0,3,Transfer Bank,2026-07-22,Cilegon,4.0,True
2,TRX00083,"Drs. Sari Aryani, M.TI.",Deserunt Basic,Olahraga,75000.0,5,Kartu Kredit,2026-09-14,Tangerang Selatan,1.0,True
3,TRX00121,"Ilsa Mahendra, S.T.",Quia Pro,Elektronik,500000.0,1,Transfer Bank,2026-08-21,Bengkulu,4.0,True
5,TRX00148,Cahyanto Agustina,Officia Basic,Buku,15000.0,5,Transfer Bank,2026-08-19,Tidore Kepulauan,4.0,True


In [15]:
# Hitung jumlah transaksi per category
print("Jumlah transaksi per kategori:")
print(df['category'].value_counts())

Jumlah transaksi per kategori:
category
Rumah Tangga    91
Kesehatan       86
Buku            81
Fashion         80
Elektronik      78
Olahraga        74
Name: count, dtype: Int64


In [18]:
from google.colab import drive
drive.mount("/content/drive")

import os
DIR_KERJA = "/content/data"                                    # sementara, cepat
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum2"       # permanen
os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))

df.to_csv(f"{DIR_SIMPAN}/transaksi_bersih.csv", index=False)
print(f"Berhasil menyimpan hasil ke Drive: {DIR_SIMPAN}/transaksi_bersih_validPrice.csv")

Mounted at /content/drive
['transaksi_bersih.csv']
Berhasil menyimpan hasil ke Drive: /content/drive/MyDrive/BigData/Praktikum2/transaksi_bersih_validPrice.csv


# Studi Kasus
Platform marketplace kita mendeteksi kejanggalan pada laporan penjualan bulanan: total transaksi yang
dilaporkan tim IT (515) tidak sama dengan total yang dipakai tim Finance (490). Sebagai calon data
engineer, jelaskan kepada tim Finance:

1. Mengapa kedua angka tersebut bisa berbeda, dikaitkan dengan proses yang baru saja Anda
lakukan.
Jawab : Karena pada tim IT data (515) total transaksi mentah belum dilakukan pemrosesan data (masih terdepat anomali pada data). Jadi data yang diserahkan ke tim Finance (490) merupakan data yang sudah bersih (tidak ada anomali lagi).

2. Apakah 490 baris “lebih benar” dibanding 515 baris? Jelaskan dengan mengaitkan ke konsep
Veracity.
Jawab : Ya, karena data yang bersih (tanpa anomali) sehingga meningkatnya kualitas dan akurasi ketika digunakan atau diambil insight nya. Kemudian sesuai dengan konsep veracity yaitu Penanganan Outlier dan Noise, serta standardisasi Format dan Konsistensi Kategori, sehingga menghasilkan data yang terpercaya dan akurat ketika digunakan.

3.  Bagaimana Anda akan menjelaskan keputusan membiarkan kolom rating tetap memiliki missing
value kepada tim Finance yang ingin tahu “rating rata-rata semua transaksi”?
Jawab : Pemberian rating oleh pelanggan merupakan opsional. Sistem otomatis mengabaikan kolom kosong, jadi hanya menghitung rata-rata dari pelanggan yang memberikan rating saja. Jika dipaksa mengisi kolom kosong dengan nilai buatan, maka datanya akan menjadi bias dan tidak akurat. Jadi, kondisi data rating ini sudah tepat untuk mencerminkan kondisi sebenarnya di lapangan.

